# Práctica 12 — Proyecto integrador y puente hacia los transformers

**Curso:** NLP Clásico con Python · **Clase 12** · **Tiempo estimado:** 90-110 minutos
**Notebook de clase:** `notebooks_clase/clase12_proyecto_puente_transformers.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Esta práctica usa las **predicciones precalculadas** del *transformer* (`datos/plan_b_transformer_resenas.json`), no descarga el modelo: así puedes hacerla sin conexión y en segundos. Si quieres ejecutar el modelo real, hazlo en el notebook de clase.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas` y `scikit-learn`.

---
## Celda 0 — Setup y datos

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

IDIOMAS = ("es", "it")   # dato-es
RESENAS = {idioma: pd.read_csv(cargar_datos(f"resenas_{idioma}.csv")) for idioma in IDIOMAS}
with open(cargar_datos("plan_b_transformer_resenas.json"), encoding="utf-8") as f:
    PLAN_B = json.load(f)
ETIQUETAS = ("negativo", "neutro", "positivo")   # dato-es

print({idioma: df.shape for idioma, df in RESENAS.items()})
print("Librerías y datos cargados.")

## Ejercicio 1 — La `Pipeline` clásica, otra vez

Completa `crear_pipeline_clasico()`: una `Pipeline` con dos pasos, `"tfidf"` (`TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)`, los mismos ajustes de las clases 5 y 6) y `"clf"` (`LogisticRegression(max_iter=1000)`).

In [ ]:
def crear_pipeline_clasico():
    return None      # TODO: Pipeline([("tfidf", ...), ("clf", ...)])

# Comprobación automática
pipe = crear_pipeline_clasico()
assert pipe is not None, "Completa las líneas marcadas con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline debe tener los pasos «tfidf» y «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2) and pipe.named_steps["tfidf"].sublinear_tf is True
assert isinstance(pipe.named_steps["clf"], LogisticRegression) and pipe.named_steps["clf"].max_iter == 1000
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline se devuelve sin ajustar"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Entrenar y evaluar, con la partición del capítulo

Completa `entrenar_y_evaluar(idioma)`: parte `RESENAS[idioma]` en 70 % entrenamiento / 30 % prueba (`train_test_split(..., test_size=0.3, stratify=df["etiqueta"], random_state=42)`), entrena `crear_pipeline_clasico()` con el entrenamiento y devuelve `(exactitud, f1_macro, prueba)` sobre la prueba, donde `prueba` es el `DataFrame` de prueba **con el índice reiniciado** (`reset_index(drop=True)`).

In [ ]:
def entrenar_y_evaluar(idioma):
    df = RESENAS[idioma]
    ent, prueba = None, None    # TODO: train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    exactitud, f1_macro = None, None    # TODO: entrena la Pipeline con ent y evalúa sobre prueba
    prueba_final = prueba.reset_index(drop=True) if prueba is not None else None
    return exactitud, f1_macro, prueba_final

# Comprobación automática
REFERENCIA = {"es": (0.722, 0.722), "it": (0.789, 0.786)}   # dato-es
for idioma in IDIOMAS:
    resultado = entrenar_y_evaluar(idioma)
    assert resultado is not None and resultado[0] is not None, "Completa las líneas marcadas con TODO"
    exactitud, f1_macro, prueba = resultado
    assert len(prueba) == 90, "30% de 300 reseñas son 90"
    ref_acc, ref_f1 = REFERENCIA[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "con esta partición y semilla, la exactitud debe coincidir con la del capítulo"
    assert abs(f1_macro - ref_f1) < 0.01, "igual para el F1 macro"
    print(idioma, "exactitud:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — La regla de mapeo de estrellas

Completa `mapear_estrellas(etiqueta_estrellas)`: recibe una cadena como `"1 star"` o `"4 stars"` y devuelve `"negativo"` (1-2), `"neutro"` (3) o `"positivo"` (4-5), según la regla justificada en el capítulo (sección 3.2). Pista: el primer carácter de la cadena es el número.

In [ ]:
def mapear_estrellas(etiqueta_estrellas):
    return None      # TODO: 1-2 -> "negativo", 3 -> "neutro", 4-5 -> "positivo"

# Comprobación automática
assert mapear_estrellas("1 star") is not None, "Completa las líneas marcadas con TODO"
assert mapear_estrellas("1 star") == "negativo" and mapear_estrellas("2 stars") == "negativo"
assert mapear_estrellas("3 stars") == "neutro"
assert mapear_estrellas("4 stars") == "positivo" and mapear_estrellas("5 stars") == "positivo"
assert [mapear_estrellas(f"{n} stars") for n in range(1, 6)] == ["negativo", "negativo", "neutro", "positivo", "positivo"]
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Evaluar el transformer con el plan B, sobre la misma prueba

Completa `evaluar_transformer(idioma, prueba)`: usa `PLAN_B["idiomas"][idioma]` para construir, **en el mismo orden que `prueba["id"]`**, la lista de estrellas precalculadas (`estrellas`) y su traducción con `mapear_estrellas`; devuelve `(exactitud, f1_macro)` comparando esas predicciones con `prueba["etiqueta"]`. `prueba` es el `DataFrame` de prueba que devuelve `entrenar_y_evaluar` (ejercicio 2): así clásico y *transformer* se comparan sobre exactamente los mismos textos.

In [ ]:
def evaluar_transformer(idioma, prueba):
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = None    # TODO: dict(zip(datos["id"], datos["estrellas"])), con las claves como texto
    estrellas = None           # TODO: una estrella por fila de prueba, en su mismo orden
    pred = None                # TODO: mapear_estrellas aplicado a cada estrella
    return None, None          # TODO: (exactitud, f1_macro)

# Comprobación automática
REFERENCIA_TRAFO = {"es": (0.856, 0.852), "it": (0.844, 0.842)}   # dato-es
for idioma in IDIOMAS:
    _, _, prueba = entrenar_y_evaluar(idioma)
    resultado = evaluar_transformer(idioma, prueba)
    assert resultado is not None and resultado[0] is not None, "Completa las líneas marcadas con TODO"
    exactitud, f1_macro = resultado
    ref_acc, ref_f1 = REFERENCIA_TRAFO[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "debe coincidir con las predicciones precalculadas del plan B, alineadas por id"
    assert abs(f1_macro - ref_f1) < 0.01
    print(idioma, "transformer exactitud:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Leer los coeficientes del clásico

Completa `top_palabras(pipe, clase, k)`: dada una `Pipeline` ya ajustada (TF-IDF + `LogisticRegression`), devuelve la lista de las `k` palabras con mayor coeficiente para `clase` (de mayor a menor), usando `pipe.named_steps["clf"].classes_` para encontrar la fila correcta.

In [ ]:
def top_palabras(pipe, clase, k=5):
    vocab = pipe.named_steps["tfidf"].get_feature_names_out()
    fila = None    # TODO: índice de "clase" en pipe.named_steps["clf"].classes_
    coef = pipe.named_steps["clf"].coef_[fila]
    return None    # TODO: las k palabras de mayor coeficiente, de mayor a menor (np.argsort)

# Comprobación automática
IDIOMA_EJ = IDIOMAS[0]   # dato-es
ent_ej, _ = train_test_split(RESENAS[IDIOMA_EJ], test_size=0.3, stratify=RESENAS[IDIOMA_EJ]["etiqueta"], random_state=42)
pipe_ej = crear_pipeline_clasico().fit(ent_ej["texto"], ent_ej["etiqueta"])
top = top_palabras(pipe_ej, "positivo", k=5)
assert top is not None, "Completa las líneas marcadas con TODO"
assert len(top) == 5, "5 palabras"
vocab_ej = pipe_ej.named_steps["tfidf"].get_feature_names_out()
coef_pos = pipe_ej.named_steps["clf"].coef_[list(pipe_ej.named_steps["clf"].classes_).index("positivo")]
valores = [coef_pos[list(vocab_ej).index(p)] for p in top]
assert valores == sorted(valores, reverse=True), "ordena de mayor a menor coeficiente"
assert top[0] == vocab_ej[int(np.argmax(coef_pos))], "la primera palabra debe tener el coeficiente más alto"
print("top positivo, idioma de ejemplo:", IDIOMA_EJ, top)
print("Ejercicio 5 correcto ✔")

---
## Reto final — cada método falla en una clase distinta

Integra lo aprendido. Completa `clase_mas_dificil(idioma)`, que debe devolver una tupla `(clase_clasico, clase_transformer)` con el nombre de la clase (`"positivo"`, `"negativo"` o `"neutro"`) con el **recall más bajo de cada método por separado** (no promediados: el reto de esta clase es descubrir que **no es la misma clase**):

- Entrena el clásico con `crear_pipeline_clasico()` sobre la misma partición 70/30 (`train_test_split(..., test_size=0.3, stratify=df["etiqueta"], random_state=42)`) y predice sobre la prueba,
- Calcula las predicciones del *transformer* con la misma lógica del ejercicio 4,
- Con `precision_recall_fscore_support(prueba["etiqueta"], predicciones, labels=ETIQUETAS)`, obtén el *recall* de cada clase para cada método y busca, en cada uno por separado, la clase con el recall más bajo (`ETIQUETAS[np.argmin(...)]`).

Comprueba que el clásico y el *transformer* fallan en clases **distintas**, y que el patrón se repite en los dos idiomas.

In [ ]:
def clase_mas_dificil(idioma):
    df = RESENAS[idioma]
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    prueba = prueba.reset_index(drop=True)
    pipe = crear_pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred_clasico = None      # TODO: pipe.predict(prueba["texto"])
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = dict(zip([str(x) for x in datos["id"]], datos["estrellas"]))
    pred_trafo = None        # TODO: mapear_estrellas de la estrella de cada id de prueba, en su orden
    recall_clasico = None    # TODO: precision_recall_fscore_support(prueba["etiqueta"], pred_clasico, labels=ETIQUETAS)[1]
    recall_trafo = None      # TODO: lo mismo con pred_trafo
    if recall_clasico is None or recall_trafo is None:
        return None
    return ETIQUETAS[int(np.argmin(recall_clasico))], ETIQUETAS[int(np.argmin(recall_trafo))]

for idioma in IDIOMAS:
    resultado = clase_mas_dificil(idioma)
    assert resultado is not None, "Completa las líneas marcadas con TODO"
    clase_clasico, clase_trafo = resultado
    assert clase_clasico is not None and clase_trafo is not None, "Completa las líneas marcadas con TODO"
    assert clase_clasico == "positivo", "según el capítulo (sección 4.1), el clásico falla más en «positivo» en los dos idiomas"
    assert clase_trafo == "neutro", "según el capítulo (sección 4.1), el transformer falla más en «neutro» en los dos idiomas"
    assert clase_clasico != clase_trafo, "el punto del reto: cada método falla en una clase distinta"
    print(idioma, "| clásico falla más en:", clase_clasico, "| transformer falla más en:", clase_trafo)
print("Reto final correcto ✔")

**Tus conclusiones (escribe 3-4 frases):** el clásico falla más en `positivo` (poco vocabulario de 210 ejemplos) y el *transformer* más en `neutro` (juicios mixtos, difíciles incluso para un modelo grande). ¿Qué te dice esto sobre combinar los dos métodos en un proyecto real? Si tuvieras que mejorar solo una clase con más datos, ¿a cuál idioma o método dedicarías el esfuerzo? ¿Qué dirías a alguien que concluye "el transformer siempre gana" a partir de esta clase?

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def crear_pipeline_clasico():
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)), ("clf", LogisticRegression(max_iter=1000))])

# Comprobación automática
pipe = crear_pipeline_clasico()
assert pipe is not None, "Completa las líneas marcadas con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline debe tener los pasos «tfidf» y «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2) and pipe.named_steps["tfidf"].sublinear_tf is True
assert isinstance(pipe.named_steps["clf"], LogisticRegression) and pipe.named_steps["clf"].max_iter == 1000
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline se devuelve sin ajustar"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def entrenar_y_evaluar(idioma):
    df = RESENAS[idioma]
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    pipe = crear_pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred = pipe.predict(prueba["texto"])
    exactitud = accuracy_score(prueba["etiqueta"], pred)
    f1_macro = f1_score(prueba["etiqueta"], pred, average="macro")
    return exactitud, f1_macro, prueba.reset_index(drop=True)

# Comprobación automática
REFERENCIA = {"es": (0.722, 0.722), "it": (0.789, 0.786)}   # dato-es
for idioma in IDIOMAS:
    resultado = entrenar_y_evaluar(idioma)
    assert resultado is not None and resultado[0] is not None, "Completa las líneas marcadas con TODO"
    exactitud, f1_macro, prueba = resultado
    assert len(prueba) == 90, "30% de 300 reseñas son 90"
    ref_acc, ref_f1 = REFERENCIA[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "con esta partición y semilla, la exactitud debe coincidir con la del capítulo"
    assert abs(f1_macro - ref_f1) < 0.01, "igual para el F1 macro"
    print(idioma, "exactitud:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def mapear_estrellas(etiqueta_estrellas):
    n = int(etiqueta_estrellas[0])
    if n <= 2:
        return "negativo"
    if n == 3:
        return "neutro"
    return "positivo"

# Comprobación automática
assert mapear_estrellas("1 star") is not None, "Completa las líneas marcadas con TODO"
assert mapear_estrellas("1 star") == "negativo" and mapear_estrellas("2 stars") == "negativo"
assert mapear_estrellas("3 stars") == "neutro"
assert mapear_estrellas("4 stars") == "positivo" and mapear_estrellas("5 stars") == "positivo"
assert [mapear_estrellas(f"{n} stars") for n in range(1, 6)] == ["negativo", "negativo", "neutro", "positivo", "positivo"]
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def evaluar_transformer(idioma, prueba):
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = dict(zip([str(x) for x in datos["id"]], datos["estrellas"]))
    estrellas = [estrellas_por_id[str(i)] for i in prueba["id"]]
    pred = [mapear_estrellas(e) for e in estrellas]
    exactitud = accuracy_score(prueba["etiqueta"], pred)
    f1_macro = f1_score(prueba["etiqueta"], pred, average="macro")
    return exactitud, f1_macro

# Comprobación automática
REFERENCIA_TRAFO = {"es": (0.856, 0.852), "it": (0.844, 0.842)}   # dato-es
for idioma in IDIOMAS:
    _, _, prueba = entrenar_y_evaluar(idioma)
    resultado = evaluar_transformer(idioma, prueba)
    assert resultado is not None and resultado[0] is not None, "Completa las líneas marcadas con TODO"
    exactitud, f1_macro = resultado
    ref_acc, ref_f1 = REFERENCIA_TRAFO[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "debe coincidir con las predicciones precalculadas del plan B, alineadas por id"
    assert abs(f1_macro - ref_f1) < 0.01
    print(idioma, "transformer exactitud:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def top_palabras(pipe, clase, k=5):
    vocab = pipe.named_steps["tfidf"].get_feature_names_out()
    fila = list(pipe.named_steps["clf"].classes_).index(clase)
    coef = pipe.named_steps["clf"].coef_[fila]
    return [vocab[i] for i in np.argsort(-coef)[:k]]

# Comprobación automática
IDIOMA_EJ = IDIOMAS[0]   # dato-es
ent_ej, _ = train_test_split(RESENAS[IDIOMA_EJ], test_size=0.3, stratify=RESENAS[IDIOMA_EJ]["etiqueta"], random_state=42)
pipe_ej = crear_pipeline_clasico().fit(ent_ej["texto"], ent_ej["etiqueta"])
top = top_palabras(pipe_ej, "positivo", k=5)
assert top is not None, "Completa las líneas marcadas con TODO"
assert len(top) == 5, "5 palabras"
vocab_ej = pipe_ej.named_steps["tfidf"].get_feature_names_out()
coef_pos = pipe_ej.named_steps["clf"].coef_[list(pipe_ej.named_steps["clf"].classes_).index("positivo")]
valores = [coef_pos[list(vocab_ej).index(p)] for p in top]
assert valores == sorted(valores, reverse=True), "ordena de mayor a menor coeficiente"
assert top[0] == vocab_ej[int(np.argmax(coef_pos))], "la primera palabra debe tener el coeficiente más alto"
print("top positivo, idioma de ejemplo:", IDIOMA_EJ, top)
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def clase_mas_dificil(idioma):
    df = RESENAS[idioma]
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    prueba = prueba.reset_index(drop=True)
    pipe = crear_pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred_clasico = pipe.predict(prueba["texto"])
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = dict(zip([str(x) for x in datos["id"]], datos["estrellas"]))
    pred_trafo = [mapear_estrellas(estrellas_por_id[str(i)]) for i in prueba["id"]]
    recall_clasico = precision_recall_fscore_support(prueba["etiqueta"], pred_clasico, labels=ETIQUETAS)[1]
    recall_trafo = precision_recall_fscore_support(prueba["etiqueta"], pred_trafo, labels=ETIQUETAS)[1]
    return ETIQUETAS[int(np.argmin(recall_clasico))], ETIQUETAS[int(np.argmin(recall_trafo))]

for idioma in IDIOMAS:
    resultado = clase_mas_dificil(idioma)
    assert resultado is not None, "Completa las líneas marcadas con TODO"
    clase_clasico, clase_trafo = resultado
    assert clase_clasico is not None and clase_trafo is not None, "Completa las líneas marcadas con TODO"
    assert clase_clasico == "positivo", "según el capítulo (sección 4.1), el clásico falla más en «positivo» en los dos idiomas"
    assert clase_trafo == "neutro", "según el capítulo (sección 4.1), el transformer falla más en «neutro» en los dos idiomas"
    assert clase_clasico != clase_trafo, "el punto del reto: cada método falla en una clase distinta"
    print(idioma, "| clásico falla más en:", clase_clasico, "| transformer falla más en:", clase_trafo)
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (zero-shot, regla de mapeo, coste computacional, explicabilidad, atención)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*